# Fusion study — 14 nhãn CheXpert (`y_*`) · CXR-only vs EHR-only vs CXR+EHR

**Câu hỏi:** *clinical context (EHR) có bổ trợ cho ảnh trong phát hiện 14 dấu hiệu X-quang không?*
Trả lời bằng **hiệu số per-nhãn**, không phải một con số tổng.

- **Target:** 14 cột `y_*` (CheXpert, per-STUDY — dấu hiệu X-quang thật của ảnh).
- **8 `dx_*` giờ là FEATURE** (tiền sử chẩn đoán, per-patient) — không còn là target.
- **3 kịch bản:** `CXR` = 768 embedding RAD-DINO · `EHR` = toàn bộ EHR (labs/vitals/microbiology + 8 `dx_`) · `CXR+EHR` = ghép cả hai.
- **Δ₁ = (CXR+EHR) − CXR** → EHR thêm được gì trên nền ảnh (đây là câu trả lời chính).
- **Δ₂ = (CXR+EHR) − EHR** → ảnh thêm được gì trên nền EHR.
- **Dữ liệu:** `fusion_dataset_hybrid_fullmicro.csv` (231,684 ảnh · 146,763 study · full microbiology 617 org + 95 spec).
- **Split TEMPORAL theo `study_dt`, đơn vị = study:** train = study sớm, test = study muộn; mọi view của cùng một study (cùng thời điểm) luôn nằm cùng phía → không rò rỉ multi-view.

> Khác biệt lớn so với bản 8-bệnh: `y_*` là nhãn **gốc của ảnh** (encoder RAD-DINO được pretrain trên chính CheXpert) → baseline `CXR` sẽ mạnh gần trần. Câu hỏi khó & đáng giá: EHR có thêm được gì **trên nền một encoder đã gần tối ưu** không.

## 0. (Colab) Mount Drive + tìm thư mục dữ liệu

In [ ]:
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('khong phai Colab / da mount:', e)

import os
CANDIDATES = [
    '/content/drive/MyDrive/MIMIC_CXR/planB_out',
    '.',
]
def _has(d):
    return (os.path.exists(os.path.join(d,'fusion_dataset_hybrid_fullmicro.parquet'))
            or os.path.exists(os.path.join(d,'fusion_dataset_hybrid_fullmicro.csv')))
OUT_DIR = next((d for d in CANDIDATES if _has(d)), CANDIDATES[0])
print('OUT_DIR =', OUT_DIR, '| co du lieu:', _has(OUT_DIR))
assert _has(OUT_DIR), 'Khong thay fusion_dataset_hybrid_fullmicro.* — sua OUT_DIR.'

## 1. Load + feature + split temporal (theo `study_dt`, đơn vị study)

In [ ]:
import pandas as pd, numpy as np, os
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# --- load fullmicro (full microbiology: 617 org_ + 95 spec_) ---
pq  = os.path.join(OUT_DIR,'fusion_dataset_hybrid_fullmicro.parquet')
csv = os.path.join(OUT_DIR,'fusion_dataset_hybrid_fullmicro.csv')
df = pd.read_parquet(pq) if os.path.exists(pq) else pd.read_csv(csv, low_memory=False)
df['study_dt'] = pd.to_datetime(df['study_dt'], errors='coerce')
print('shape:', df.shape, '| study_dt nan:', int(df['study_dt'].isna().sum()))

# --- TARGET: 14 nhan CheXpert ---
FINDINGS = [c for c in df.columns if c.startswith('y_')]
NAMES    = [c[2:] for c in FINDINGS]
print('FINDINGS (14):', NAMES)

# --- FEATURE ---
IMG = [f'img_{i}' for i in range(768)]
DROP_EXACT = {'dicom_id','subject_id','study_id','hadm_id','stay_id','first_hadm_id',
              'first_admittime','first_icd10_codes','study_dt','admittime','ed_intime',
              'encounter_time','split','view','disease_group','chiefcomplaint'}
# GIU dx_* lam feature; loai y_* (target), img_, va meta
EHR = [c for c in df.columns if not c.startswith(('img_','y_')) and c not in DROP_EXACT]
if 'gender' in df.columns:
    df['gender'] = df['gender'].map({'M':1,'F':0})

# --- (tuy chon) loc org_/spec_ qua thua: bo cot co < MIN_POS ca duong tren toan bo ---
# ly do: duoi temporal split, mot loai vi khuan hiem co the chi xuat hien o test -> nhieu/rò rỉ chieu.
MIN_POS = 10          # dat 0 de giu het 712 cot microbiology
sparse = [c for c in EHR if c.startswith(('org_','spec_'))]
if MIN_POS > 0:
    pos = {c: int(pd.to_numeric(df[c], errors='coerce').fillna(0).astype(bool).sum()) for c in sparse}
    keep = [c for c in sparse if pos[c] >= MIN_POS]
    EHR = [c for c in EHR if not c.startswith(('org_','spec_'))] + keep
    print(f'microbiology: giu {len(keep)}/{len(sparse)} cot (MIN_POS={MIN_POS})')
print('IMG:', len(IMG), '| EHR:', len(EHR), '| dx_ trong EHR:', sum(c.startswith('dx_') for c in EHR))

# --- SPLIT TEMPORAL, don vi = study (moi view cung study_dt di cung nhau) ---
TEST_FRAC = 0.2
stime = df.groupby('study_id')['study_dt'].min().sort_values()   # 1 study = 1 moc thoi gian
cut   = int(round((1-TEST_FRAC)*len(stime)))
train_studies = set(stime.index[:cut]); test_studies = set(stime.index[cut:])
cut_time = stime.iloc[cut]
tr = df['study_id'].isin(train_studies).values
te = df['study_id'].isin(test_studies).values
assert not (train_studies & test_studies)                            # 0 study nam 2 ben
assert df.loc[tr,'study_dt'].max() <= df.loc[te,'study_dt'].min()     # train truoc test
print(f'cut_time = {cut_time}')
print(f'train: {tr.sum()} anh / {len(train_studies)} study | test: {te.sum()} anh / {len(test_studies)} study')
print('n_pos_test:', {n:int(df.loc[te,c].sum()) for n,c in zip(NAMES,FINDINGS)})

def prep(cols):                                   # impute(median)+scale, FIT TREN TRAIN
    X  = df[cols].apply(pd.to_numeric, errors='coerce').to_numpy('float32')
    Xi = SimpleImputer(strategy='median').fit(X[tr]).transform(X)
    return StandardScaler().fit(Xi[tr]).transform(Xi).astype('float32')

FEATSETS = {'CXR': IMG, 'EHR': EHR, 'CXR+EHR': IMG+EHR}
Xcache = {n: prep(c) for n,c in FEATSETS.items()}
print('prep xong:', {k:v.shape for k,v in Xcache.items()})

## 2. Train — 2 model × 3 kịch bản × 14 nhãn (84 lần fit)

- **LogReg** = *linear probe* trên embedding: baseline ảnh **đúng & mạnh** cho nhãn CheXpert (encoder RAD-DINO pretrain sao cho các nhãn này tách tuyến tính). XGBoost cây nông thường **underfit** embedding dense → thấp hơn LogReg ở các nhãn này.
- **XGBoost** = giữ để đối chứng với pipeline 8-bệnh cũ.
- Ngưỡng cố định **0.5**, KHÔNG xử lý imbalance → **F1/recall phụ thuộc ngưỡng**; muốn so công bằng với bảng báo AUROC thì nhìn **AUROC/AUPRC** (độc lập ngưỡng), đừng nhìn F1.
- LogReg nhanh (~vài giây/fit); XGBoost là phần lâu.

In [ ]:
import time, os, numpy as np, pandas as pd
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             fbeta_score, average_precision_score, roc_auc_score)

def build_xgb():        # them device='cuda' neu co GPU (xgboost>=2.0)
    return XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                         subsample=0.85, colsample_bytree=0.85, tree_method='hist',
                         eval_metric='logloss', n_jobs=-1, random_state=0)
def build_logreg():     # LINEAR PROBE: hop voi embedding CheXpert (encoder pretrain tuyen tinh)
    return LogisticRegression(max_iter=2000, C=1.0, n_jobs=-1)

# LogReg = baseline anh dung/manh cho nhan CheXpert; XGBoost = giu doi chung voi pipeline cu
MODELS = {'LogReg': build_logreg, 'XGBoost': build_xgb}
THRESH = 0.5            # nguong co dinh; F1/recall o day PHU THUOC nguong -> so AUROC/AUPRC (doc lap nguong)

tr_idx = np.where(tr)[0]; te_idx = np.where(te)[0]
rows=[]
for mname, mk in MODELS.items():
    for fname in FEATSETS:
        Xtr, Xte = Xcache[fname][tr_idx], Xcache[fname][te_idx]
        for name, ycol in zip(NAMES, FINDINGS):
            y = df[ycol].values.astype(int); ytr, yte = y[tr_idx], y[te_idx]
            t0=time.time(); clf = mk().fit(Xtr, ytr)
            proba = clf.predict_proba(Xte)[:,1]; pred=(proba>=THRESH).astype(int)
            auroc = roc_auc_score(yte, proba) if yte.min()!=yte.max() else float('nan')
            r = dict(model=mname, featset=fname, finding=name,
                     accuracy=accuracy_score(yte,pred),
                     precision=precision_score(yte,pred,zero_division=0),
                     recall=recall_score(yte,pred,zero_division=0),
                     f1=f1_score(yte,pred,zero_division=0),
                     f2=fbeta_score(yte,pred,beta=2,zero_division=0),
                     auroc=auroc, auprc=average_precision_score(yte,proba),
                     prevalence=float(yte.mean()), n_pos_test=int(yte.sum()))
            rows.append(r)
            print(f"[{mname:7}|{fname:8}] {name:26} f1={r['f1']:.3f} auroc={auroc:.3f} auprc={r['auprc']:.3f} ({time.time()-t0:.0f}s)")

results = pd.DataFrame(rows)
results.to_csv(os.path.join(OUT_DIR,'benchmark_14findings.csv'), index=False)
print('luu benchmark_14findings.csv | dong:', len(results), '(ky vong 84)')
results.head(14)

## 3. Heatmap 3 kịch bản + Δ

- 3 bảng điểm: **CXR**, **EHR**, **CXR+EHR** (cùng thang 0–1).
- **Δ₁ = (CXR+EHR) − CXR**: đỏ = EHR **giúp** trên nền ảnh · xanh = EHR làm hại.
- **Δ₂ = (CXR+EHR) − EHR**: đỏ = ảnh **giúp** trên nền EHR.

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns, os, pandas as pd
METRICS = ['precision','recall','f1','f2','auroc','auprc']   # auroc de so voi bang bao AUROC
res = pd.read_csv(os.path.join(OUT_DIR,'benchmark_14findings.csv'))
ORDER = (res[res.featset=='CXR'].groupby('finding')['prevalence'].first()
         .sort_values(ascending=False).index.tolist())

for mname in res.model.unique():
    sub = res[res.model==mname]
    def panel(fs): return sub[sub.featset==fs].set_index('finding')[METRICS].reindex(ORDER)
    A, E, B = panel('CXR'), panel('EHR'), panel('CXR+EHR')

    fig, ax = plt.subplots(1, 3, figsize=(26, 6))
    for a, M, ti in [(ax[0],A,f'{mname} — CXR'), (ax[1],E,f'{mname} — EHR'), (ax[2],B,f'{mname} — CXR+EHR')]:
        sns.heatmap(M, ax=a, cmap='RdYlGn', vmin=0, vmax=1, annot=True, fmt='.3f',
                    linewidths=.5, linecolor='black', annot_kws={'size':8}, cbar_kws={'label':'score'})
        a.set_title(ti); a.set_xlabel(''); a.set_ylabel('finding')
        a.set_yticklabels(a.get_yticklabels(), rotation=0)
    plt.tight_layout(); plt.show()

    D1, D2 = B - A, B - E
    fig, ax = plt.subplots(1, 2, figsize=(18, 6))
    for a, D, ti in [(ax[0],D1,f'{mname}  Δ₁=(CXR+EHR)−CXR  (đỏ=EHR giúp)'),
                     (ax[1],D2,f'{mname}  Δ₂=(CXR+EHR)−EHR  (đỏ=ảnh giúp)')]:
        sns.heatmap(D, ax=a, cmap='RdBu_r', center=0, vmin=-.3, vmax=.3, annot=True, fmt='+.3f',
                    linewidths=.5, linecolor='black', annot_kws={'size':8}, cbar_kws={'label':'Δ'})
        a.set_title(ti); a.set_xlabel(''); a.set_ylabel('finding')
        a.set_yticklabels(a.get_yticklabels(), rotation=0)
    plt.tight_layout(); plt.show()

# --- AUPRC la chi so chinh (hop mat can bang). Doc kem 'lift' = auprc/prevalence (san ngau nhien = prevalence) ---
piv  = res.pivot_table(index='finding', columns=['model','featset'], values='auprc').reindex(ORDER)
prev = res.groupby('finding')['prevalence'].first().reindex(ORDER)
print('=== AUPRC (san ngau nhien = prevalence) ===')
print(piv.round(3).to_string())
print('\\n=== AUPRC / prevalence  (lift tren ngau nhien; >1 la co skill) ===')
print(piv.div(prev, axis=0).round(2).to_string())
print('\\nprevalence:', {k:round(v,3) for k,v in prev.items()})

## Đọc kết quả

- **So sánh với bảng ngoài phải cùng thang metric.** Bảng báo **AUROC** thì so với cột **auroc** ở đây, KHÔNG so với AUPRC (AUPRC luôn thấp hơn nhiều khi mất cân bằng). Và **đừng so F1** trực tiếp: F1 ở đây tính tại ngưỡng 0.5 không cân bằng, model recall-optimized (tune ngưỡng/class-weight) sẽ có F1 cao hơn dù ranking không tốt hơn.
- **`-1` (uncertain):** dataset này `y_*` đã là 0/1 (U-policy áp sẵn ở bước build). Nếu pipeline khác resolve `-1` khác đi → task khác → không so được. Kiểm tra policy trước khi so.
- **Δ₁ (cột chính):** nhãn nào EHR bổ trợ được cho ảnh. Giả thuyết: các dấu hiệu liên quan sinh lý/xét nghiệm (Edema, Cardiomegaly, Pleural Effusion, Pneumonia) được labs/vitals/culture giúp; các dấu hiệu thuần hình thái (Fracture, Lung Lesion, Pneumothorax) gần như không.
- **Ceiling effect:** vì RAD-DINO pretrain trên CheXpert, `CXR` mạnh sẵn → Δ₁ dương dù nhỏ vẫn là bằng chứng EHR mang thông tin **bổ sung**, không trùng ảnh.
- **AUPRC vs `prevalence`:** chỉ số trung thực khi mất cân bằng; so AUPRC với prevalence để biết hơn đoán mù bao nhiêu.
- **Temporal split:** train = quá khứ, test = tương lai → phản ánh triển khai thực tế; đơn vị study nên không rò rỉ multi-view. Lưu ý: một bệnh nhân *có thể* xuất hiện cả train lẫn test qua 2 lần khám khác thời điểm — đây là hợp lệ trong temporal validation (không phải leakage). Muốn chặt hơn (patient-disjoint), đổi khóa split sang `subject_id` theo thời gian.
- **Mở rộng:** thêm LightGBM để cross-check; lặp nhiều seed để ra mean±std như heatmap gốc; tune ngưỡng per-nhãn trên một khối val thời-gian-giữa nếu muốn recall cao hơn.